In [1]:
pip install sympy

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install scienceplots

  Using cached scienceplots-2.2.2-py3-none-any.whl.metadata (14 kB)
Using cached scienceplots-2.2.2-py3-none-any.whl (33 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sympy as sp
from sympy.physics.quantum.dagger import Dagger
from sympy.physics.quantum import TensorProduct
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LogNorm, Normalize
from sympy.printing.mathematica import mathematica_code
from sympy.parsing.mathematica import parse_mathematica
import scipy
import scienceplots


sp.init_printing()
plt.rcParams['figure.dpi'] = 400

In [42]:
t = sp.Symbol('t')
h_bar = sp.Symbol("hbar")

plt.style.use('science')

def commutator(A,B):
    return A@B - B@A

def anti_commutator(A,B):
    return A@B + B@A

def L_i_j(i,j, N): # jump operators
    return sp.Matrix(N, N, lambda m, n: 1 if (n==i and m==j) else 0) 


# if H is specified, then the lindbladian will be computed using that H
# otherwise, a generic H is built
# includes the dissipator, unless specified false
def lindbladian(N, output=False, degenerate=False, H=sp.Matrix([0]), L_operators=[], dissipative=True):
    # unitary time evolution part
    
    rho = sp.Matrix(N, N, lambda i, j: sp.Function(f"rho{i}{j}")(t)) # density matrix
    
    if H == sp.Matrix([0]):
        H0 = sp.Matrix(N, N, lambda i, j: sp.Symbol(f"epsilon{i}{j}") if i==j else 0) # unperturbed hamiltonian
        # setting levels degenerate
        if degenerate: H0 = H0.subs({f"epsilon{i}{i}":0 for i in range(N)})

        V = sp.Matrix(N, N, lambda i, j: sp.Symbol(f"V{i}{j}") if i!=j else 0) # perturbation

        # since V is hermitian
        V = V.subs({sp.Symbol(f"V{j}{i}"):sp.Symbol(f"V{i}{j}").conjugate() for i in range(N) for j in range(i)})


        H = H0 + V

    if output: H

    
    
    #---------------------------------------------------------------------------------------------
    # computing dissipator
    D = dissipator(N, L_operators) if dissipative else sp.zeros(N)
    if output:
        display(H)
        display(D)


    #---------------------------------------------------------------------------------------------
    # computing lindbladian
    P = (-sp.I*commutator(H,rho) + D) # louisvillian superoperator

    eqs = [P[i, j] for i in range(N) for j in range(N)] 

    rho = [sp.Function(f"rho{i}{j}")(t) for i in range(N) for j in range(N)]

    L, _ = sp.linear_eq_to_matrix(eqs, rho)
    
    rho = sp.Matrix(rho).reshape(N**2,1)
    
    
    if output: L
    
    return L, rho


# computes generic dissipator for system with N sites
def dissipator(N, L_operators):
    # non unitary / dissipation evolution
    rho = sp.Matrix(N, N, lambda i, j: sp.Function(f"rho{i}{j}")(t)) # density matrix
    D = sp.zeros(N,N) # dissipator
    
    if not L_operators:
        # population transitions
        for i in range(N):
            for j in range(N):
                if i != j: # avoids double counting
                    rate_const_gamma = sp.Symbol(f"Gamma_{i}→{j}", real=True, nonnegative=True)
                    D += rate_const_gamma*(L_i_j(i,j,N)*rho*Dagger(L_i_j(i,j,N)) - anti_commutator(Dagger(L_i_j(i,j,N))*L_i_j(i,j,N), rho)*sp.Rational(1,2))
                    #D += simplify(L_i_j(i,j)*rho*Dagger(L_i_j(i,j)))

        # dephasing
        for i in range(N):
            rate_const_gamma = sp.Symbol(f"Gamma_{i}→{i}", real=True, nonnegative=True)
            D += rate_const_gamma*(L_i_j(i,i,N)*rho*Dagger(L_i_j(i,i,N)) - anti_commutator(Dagger(L_i_j(i,i,N))*L_i_j(i,i,N), rho)*sp.Rational(1,2))
    else:
        for i, operator in enumerate(L_operators):
            rate_const_gamma = sp.Symbol(f"Gamma_{i}", real=True, nonnegative=True)
            D += rate_const_gamma*(operator*rho*Dagger(operator) - anti_commutator(Dagger(operator)*operator, rho)*sp.Rational(1,2))
            
            
            
    return D






# initial conditions is a list containing the inital (t=0) values of each value of rho
def solve_lindblad_ana(L, rho, initial_conditions = []):
    if not initial_conditions: initial_conditions = [rho[i].subs(t, 0) for i in range(len(rho))]
    return sp.dsolve([sp.Eq(rho.diff()[i], (L@rho)[i]) for i in range(len(rho))], 
                     ics={rho[i].subs(t, 0): initial_conditions[i] for i in range(len(rho))},
                    hint="best",
                     x0=0,
                     simplify=False
                    )



# takes in an expr in terms of independent_variable, start/stop times, and time step
# returns the times array and the right-hand side of the equation evaluted at each time step
def compute_eq_ana(expr, independent_variable, start, stop, dt):
    times = np.arange(start, stop, dt)
    
    f = sp.lambdify(independent_variable, expr)
    
    return times, np.full_like(times, f(times.astype(complex)), dtype=complex)
        


def solve_lindblad_num(L, start, stop, dt, initial_conditions):
    times = np.arange(start, stop, dt)
    
    vals = [scipy.linalg.expm(np.array(L).astype(complex)*time)@np.array(initial_conditions) for time in times]
    
    return times, np.array(vals).T
    

def plot_spectrum(expr, independent_variable, start, stop, delta, color="viridis", label="", size=2, norm="linear", ax="", fig="", title=""):
    if not ax or not fig:
        fig, ax = plt.subplots()
        print("xxxxxxxxxxx")
    
    steps = np.arange(start, stop, delta) if norm=="linear" else np.logspace(np.log10(start), np.log10(stop), 200)
    scale = LogNorm(vmin=start, vmax=stop) if norm=="log" else Normalize(vmin=start, vmax=stop)
    
    f = sp.lambdify(sp.Symbol(independent_variable), expr)
    

    val = f(steps.astype(complex))
    ax.scatter(np.real(val), np.imag(val), s=size, c=steps, cmap=color, label=label, norm=scale)

    
    fig.colorbar(cm.ScalarMappable(norm=scale, cmap=color), ax=ax).set_label(label)
    ax.grid()
    ax.set_title(title)
    ax.set_axisbelow(True)
    ax.set_xlabel("Real Axis")
    ax.set_ylabel("Imag Axis")


# defining pauli matrices
sigma_x = sp.Matrix([[0, 1], [1, 0]])
sigma_y = sp.Matrix([[0, -sp.I], [sp.I, 0]])
sigma_z = sp.Matrix([[1, 0], [0, -1]])
sigma_0 = sp.eye(2)

# returns a version of the entered sigma matrix that acts on the ith state, n is the number of states
# sigma_i = I x I x ... I x sigma x I x ... I
def sigma(sigma, i, n):
    sigma_i = sigma
    
    chain = [sigma_0 for i in range(i)]
    
    chain.append(sigma)
    
    chain.extend([sigma_0 for i in range(n-i-1)])
    
    return TensorProduct(*chain)


    
# returns the raising operator that raises the ith level of n total levels (0 indexed)
def sigma_plus(i, n):
    return (sigma(sigma_x, i, n) + sp.I*sigma(sigma_y, i, n))*sp.Rational(1/2)

# returns the lowering operator that lowers the ith level of n total levels (0 indexed)
def sigma_minus(i, n):
    return (sigma(sigma_x, i, n) - sp.I*sigma(sigma_y, i, n))*sp.Rational(1/2)
